# Notebook 04: Reproducible reporting

**Introduction to AI-Assisted Research Methods and Data Analysis** · Day 4

NexusHealth · Adamu Onu

This notebook rebuilds every reported number, table and figure from the analysis set, writes them to files, and renders the mini research report. If someone changes the data, one **Restart and Run All** should regenerate everything.

> **Teaching data only.** Every patient in this course is synthetic. The associations are simulated and are not clinical evidence. Never paste identifiable patient data, confidential manuscripts or restricted full text into a public AI tool.

### How to use this notebook

1. Run cells from top to bottom with **Shift + Enter**.
2. Before you run a cell, say out loud (or write) what you expect it to do. After it runs, check whether you were right. We call this *predict, run, inspect*.
3. Cells marked **Exercise** have gaps for you to fill. Cells marked **AI assist** suggest a prompt for GitHub Copilot Chat (or your own assistant).
4. If things get tangled, use **Restart** then **Run All** from the notebook toolbar.

# Part A: A reproducible workflow (block 1)

A result is reproducible when someone else can regenerate it from your files without asking you anything. Five habits get you most of the way:

1. **Run All from a fresh start.** Hidden state (a variable you made, then deleted the cell) is the most common reason notebooks break.
2. **Relative paths only**, built from the course folder.
3. **Assertions** that check the things you rely on.
4. **Outputs written by code**, never copied and pasted by hand.
5. **A record** of software versions and of how AI was used.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the course folder, wherever this notebook lives inside it.
ROOT = Path.cwd()
while not (ROOT / "data" / "raw").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "raw").is_dir(), "Course folder not found. Open the whole course folder in VS Code."

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_REFERENCE = ROOT / "data" / "reference"
OUT_TABLES = ROOT / "outputs" / "tables"
OUT_FIGURES = ROOT / "outputs" / "figures"
for folder in (DATA_PROCESSED, OUT_TABLES, OUT_FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
print("Course folder found:", ROOT.name)

In [ ]:
analysis_path = DATA_PROCESSED / "analysis_set.csv"
if not analysis_path.exists():
    analysis_path = DATA_REFERENCE / "analysis_set_reference.csv"
    print("Your Day 2 file was not found, so the reference analysis set is used instead.")
df = pd.read_csv(analysis_path, parse_dates=["visit_date"])

# Tell Python which variables are categories, and which level is the reference.
df["sex"] = pd.Categorical(df["sex"], categories=["Female", "Male"])
df["smoking_status"] = pd.Categorical(df["smoking_status"], categories=["Never", "Former", "Current"])
df["clinic"] = pd.Categorical(df["clinic"], categories=["North", "East", "South", "West"])
print(f"Loaded {analysis_path.name}: {df.shape[0]} rows, {df.shape[1]} columns")

## A1. Check what we rely on

If any of these fail, stop and find out why before reporting anything.

In [ ]:
MODEL_VARS = ["systolic_bp", "uncontrolled_bp", "adherence_10", "age", "sex", "bmi", "diabetes", "smoking_status"]
assert df["patient_id"].is_unique, "Duplicate patients in the analysis set"
assert df[MODEL_VARS].notna().all().all(), "Missing values in model variables"
assert df["adherence_10"].between(0, 10).all(), "Adherence outside 0 to 100"
assert set(df["uncontrolled_bp"]) <= {0, 1}
print("Checks passed for", len(df), "patients.")

## A2. Record the software environment

In [ ]:
import platform
import matplotlib, scipy, seaborn, statsmodels

session = {
    "date_run": str(pd.Timestamp.now().round("s")),
    "python": platform.python_version(),
    **{m.__name__: m.__version__ for m in (np, pd, scipy, statsmodels, matplotlib, seaborn)},
}
(ROOT / "outputs" / "logs").mkdir(parents=True, exist_ok=True)
pd.Series(session).to_csv(ROOT / "outputs" / "logs" / "session_info.csv", header=["version"])
pd.Series(session)

## Exercise: Reproducibility audit

Open Notebook 02 and Notebook 03. For each: **Restart**, **Run All**, and note any error. Then answer: (1) Did both run without errors? (2) Did any number change from yesterday? (3) Find one place where a result depends on something outside the notebook (a file, a setting, a manual step) and say how it is protected.

**Your answer:** _Double-click here and write your answer._

# Part B: Tables, figures and the report (block 2)

## B1. Refit the models

This notebook fits the models again so that it can run on its own.

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

COVARIATES = "age + sex + bmi + C(diabetes) + smoking_status"
models = {
    "lin_crude": smf.ols("systolic_bp ~ adherence_10", data=df).fit(),
    "lin_adj": smf.ols(f"systolic_bp ~ adherence_10 + {COVARIATES}", data=df).fit(),
    "log_crude": smf.logit("uncontrolled_bp ~ adherence_10", data=df).fit(disp=False),
    "log_adj": smf.logit(f"uncontrolled_bp ~ adherence_10 + {COVARIATES}", data=df).fit(disp=False),
}

def effect(model, term="adherence_10", exponentiate=False):
    est, (low, high) = model.params[term], model.conf_int().loc[term]
    if exponentiate:
        est, low, high = np.exp([est, low, high])
    return est, low, high, model.pvalues[term]

def fmt_p(p):
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

for name, model in models.items():
    est, low, high, p = effect(model, exponentiate=name.startswith("log"))
    print(f"{name:<10} {est:6.2f} ({low:.2f} to {high:.2f}), p {fmt_p(p)}")

## B2. Table 1: characteristics of the analysis set

Table 1 describes who was studied. We split it by outcome status, and use mean (SD) for roughly symmetric variables, median (IQR) for skewed ones, and n (%) for categories. We do **not** add p-values to Table 1: its job is description.

In [ ]:
def mean_sd(s):
    return f"{s.mean():.1f} ({s.std():.1f})"

def median_iqr(s):
    s = s.dropna()
    return f"{s.median():.0f} ({s.quantile(0.25):.0f} to {s.quantile(0.75):.0f})"

def n_pct(s, value):
    return f"{(s == value).sum()} ({100 * (s == value).mean():.1f})"

groups = {
    f"All (n = {len(df)})": df,
    f"Controlled (n = {(df['uncontrolled_bp'] == 0).sum()})": df[df["uncontrolled_bp"] == 0],
    f"Uncontrolled (n = {(df['uncontrolled_bp'] == 1).sum()})": df[df["uncontrolled_bp"] == 1],
}
rows = [
    ("Age, years, mean (SD)", lambda d: mean_sd(d["age"])),
    ("Female, n (%)", lambda d: n_pct(d["sex"], "Female")),
    ("BMI, kg/m², mean (SD)", lambda d: mean_sd(d["bmi"])),
    ("Diabetes, n (%)", lambda d: n_pct(d["diabetes"], 1)),
    ("Smoking: never, n (%)", lambda d: n_pct(d["smoking_status"], "Never")),
    ("Smoking: former, n (%)", lambda d: n_pct(d["smoking_status"], "Former")),
    ("Smoking: current, n (%)", lambda d: n_pct(d["smoking_status"], "Current")),
    ("Medication adherence, %, mean (SD)", lambda d: mean_sd(d["medication_adherence"])),
    ("Physical activity, min/week, median (IQR)†", lambda d: median_iqr(d["physical_activity_min"])),
    ("Systolic BP, mmHg, mean (SD)", lambda d: mean_sd(d["systolic_bp"])),
    ("Diastolic BP, mmHg, mean (SD)", lambda d: mean_sd(d["diastolic_bp"])),
]
table1 = pd.DataFrame({name: [f(data) for _, f in rows] for name, data in groups.items()},
                      index=[label for label, _ in rows]).rename_axis("Characteristic")
n_missing_pa = df["physical_activity_min"].isna().sum()
table1_note = f"†Physical activity missing for {n_missing_pa} patients. Uncontrolled BP: SBP ≥ 140 or DBP ≥ 90 mmHg (course definition)."
table1

## B3. Table 2: the main results

A focused results table answers the research question directly: the adherence estimate, crude and adjusted, for both outcomes. The full adjusted models go in a supplementary table.

In [ ]:
def est_ci(model, exponentiate=False, digits=2):
    est, low, high, _ = effect(model, exponentiate=exponentiate)
    return f"{est:.{digits}f} ({low:.{digits}f} to {high:.{digits}f})"

table2 = pd.DataFrame({
    "Outcome": ["Systolic BP (mmHg)", "Uncontrolled BP"],
    "Effect measure": ["Mean difference", "Odds ratio"],
    "Crude (95% CI)": [est_ci(models["lin_crude"]), est_ci(models["log_crude"], True)],
    "Adjusted (95% CI)": [est_ci(models["lin_adj"]), est_ci(models["log_adj"], True)],
    "Adjusted p": [fmt_p(effect(models["lin_adj"])[3]), fmt_p(effect(models["log_adj"])[3])],
})
table2_note = (f"Estimates per 10-point higher adherence score; n = {len(df)}. Adjusted for age, sex, BMI, "
               "diabetes and smoking status. Wald 95% confidence intervals.")

# Supplementary table: every term in both adjusted models
labels = {"Intercept": "Intercept", "adherence_10": "Adherence (per 10 points)", "age": "Age (per year)",
          "sex[T.Male]": "Male (vs female)", "bmi": "BMI (per kg/m²)", "C(diabetes)[T.1]": "Diabetes (vs none)",
          "smoking_status[T.Former]": "Former smoker (vs never)", "smoking_status[T.Current]": "Current smoker (vs never)"}
tableS1 = pd.DataFrame({
    "Term": [labels[t] for t in models["lin_adj"].params.index],
    "SBP difference, mmHg (95% CI)": [f"{e[0]:.2f} ({e[1]:.2f} to {e[2]:.2f})"
                                      for e in (effect(models["lin_adj"], t) for t in models["lin_adj"].params.index)],
    "Odds ratio (95% CI)": [f"{e[0]:.2f} ({e[1]:.2f} to {e[2]:.2f})"
                            for e in (effect(models["log_adj"], t, True) for t in models["log_adj"].params.index)],
})
# Same row order as the R version, without the intercept
tableS1 = tableS1.set_index("Term").loc[[v for v in labels.values() if v != "Intercept"]].reset_index()
display(table2)
tableS1

## B4. The main figure

A good results figure shows the estimate *and* its uncertainty on a scale readers understand. Here: the predicted probability of uncontrolled BP across the adherence range, for a reference patient (55 years old, female, BMI 28.5, no diabetes, never smoked), with a 95% confidence band. We save it at 300 dpi and write a text description for readers who cannot see it.

In [ ]:
reference = pd.DataFrame({
    "adherence_10": np.linspace(1, 10, 91), "age": 55, "sex": "Female", "bmi": 28.5,
    "diabetes": 0, "smoking_status": "Never",
})
reference["sex"] = pd.Categorical(reference["sex"], categories=["Female", "Male"])
reference["smoking_status"] = pd.Categorical(reference["smoking_status"], categories=["Never", "Former", "Current"])

# Predictions on the log-odds scale, then converted to probabilities
log_adj = models["log_adj"]
linear = log_adj.get_prediction(reference, which="linear").summary_frame()
expit = lambda x: 1 / (1 + np.exp(-x))
reference["prob"] = expit(linear["predicted"])
reference["low"] = expit(linear["ci_lower"])
reference["high"] = expit(linear["ci_upper"])

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.fill_between(reference["adherence_10"] * 10, reference["low"], reference["high"], color="#A2B5CD", alpha=0.5,
                label="95% confidence band")
ax.plot(reference["adherence_10"] * 10, reference["prob"], color="#002366", linewidth=2.5, label="Predicted probability")
ax.set(xlabel="Medication adherence score (%)", ylabel="Probability of uncontrolled BP",
       ylim=(0, 1), title="Predicted probability of uncontrolled BP by adherence")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(frameon=False, loc="lower left")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_FIGURES / "figure1_predicted_probability.png", dpi=300)
plt.show()

at = lambda score: reference.loc[np.isclose(reference["adherence_10"], score / 10)].iloc[0]
figure_alt = (f"Line chart. For a reference patient, the predicted probability of uncontrolled blood pressure "
              f"falls from {at(20)['prob']:.0%} at an adherence score of 20 to {at(100)['prob']:.0%} at 100. "
              f"The confidence band is widest at low adherence, where there are fewer patients.")
print(figure_alt)

## B5. Write everything the report needs

The report never contains a hand-typed result. Tables are written as Markdown files, and the key numbers are written to `reports/_variables.yml`, which the report reads. Change the data, run this notebook again, and the report text updates itself.

In [ ]:
def write_md_table(table, path, note):
    path.write_text(table.to_markdown(index=False) + f"\n\n*{note}*\n", encoding="utf-8")

write_md_table(table1.reset_index(), OUT_TABLES / "table1.md", table1_note)
write_md_table(table2, OUT_TABLES / "table2.md", table2_note)
write_md_table(tableS1, OUT_TABLES / "tableS1.md", "Adjusted models; Wald 95% confidence intervals.")
table1.to_csv(OUT_TABLES / "table1.csv")
table2.to_csv(OUT_TABLES / "table2.csv", index=False)

lin_adj, log_adj_e = effect(models["lin_adj"]), effect(models["log_adj"], exponentiate=True)
lin_crude, log_crude_e = effect(models["lin_crude"]), effect(models["log_crude"], exponentiate=True)
counterfactual = {s: models["log_adj"].predict(df.assign(adherence_10=s / 10)).mean() for s in (40, 80)}
variables = {
    "n_analysis": len(df),
    "n_clean": int(pd.read_csv(OUT_TABLES / "analysis_set_flow.csv")["n_remaining"].iloc[0])
               if (OUT_TABLES / "analysis_set_flow.csv").exists() else 640,
    "pct_uncontrolled": f"{100 * df['uncontrolled_bp'].mean():.1f}",
    "lin_crude": f"{lin_crude[0]:.2f} mmHg (95% CI {lin_crude[1]:.2f} to {lin_crude[2]:.2f})",
    "lin_adj": f"{lin_adj[0]:.2f} mmHg (95% CI {lin_adj[1]:.2f} to {lin_adj[2]:.2f})",
    "log_crude": f"{log_crude_e[0]:.2f} (95% CI {log_crude_e[1]:.2f} to {log_crude_e[2]:.2f})",
    "log_adj": f"{log_adj_e[0]:.2f} (95% CI {log_adj_e[1]:.2f} to {log_adj_e[2]:.2f})",
    "prob_40": f"{100 * counterfactual[40]:.0f}%",
    "prob_80": f"{100 * counterfactual[80]:.0f}%",
    "figure_alt": figure_alt,
}
lines = [f'{key}: "{value}"' for key, value in variables.items()]
(ROOT / "reports" / "_variables.yml").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

## Exercise 4A: Rewrite weak results prose

A first draft of the results said:

> *"Adherence was highly significant (p<0.001) for blood pressure, proving that taking medication lowers BP. Patients with good adherence had 19% less uncontrolled BP."*

**List what is wrong, then rewrite it** in two or three sentences using the numbers above.

**Your answer:** _Double-click here and write your answer._

## B6. Render the report

The report lives in `reports/mini_report.qmd`. It is a Quarto document: plain text with headings, plus instructions to include the tables and figure written above. The next cell asks Quarto to turn it into an HTML page and a Word document. If Quarto is not installed, use the instructions printed below, or render it from the VS Code command palette (**Quarto: Render**).

In [ ]:
import shutil
import subprocess

if shutil.which("quarto"):
    for fmt in ("html", "docx"):
        result = subprocess.run(["quarto", "render", "mini_report.qmd", "--to", fmt],
                                cwd=ROOT / "reports", capture_output=True, text=True)
        print(fmt, "rendered" if result.returncode == 0 else "FAILED\n" + result.stderr[-2000:])
    print("Open reports/mini_report.html in your browser.")
else:
    print("Quarto was not found. Install it from https://quarto.org, then run in a terminal:\n"
          "  cd reports && quarto render mini_report.qmd")

# Part C: Structured discussion (block 3)

Use five labelled parts. Draft each in the discussion matrix (`templates/Discussion_Matrix_Template.docx`) before writing prose.

| Label | Part | Question it answers |
|---|---|---|
| **D** | Direct answer | What did we find, in one or two sentences, with the main estimate? |
| **L** | Literature | How does this compare with the verified sources in our literature matrix? |
| **F** | Framework | What does our conceptual or theoretical framework say this could mean? What did we *not* test? |
| **A** | Alternative explanations and limitations | Confounding, selection, measurement, missing data. For each: the likely direction and size of its effect. |
| **R** | Recommendation and next step | What should practice or the next study do, given the strength of evidence? |

**Claim and evidence audit.** For every sentence in your discussion that states a fact, write which evidence supports it: a cell in this notebook, a row in your literature matrix (with the exact passage), or *none*. Delete or soften any claim marked *none*.

## Exercise: Draft the D and A parts

Write the **D** part and one **A** limitation for the teaching case. For the limitation, name the problem, the likely direction of bias, and what would fix it.

**Your answer:** _Double-click here and write your answer._

# Part D: Capstone handover (block 4)

The handover package must let a reviewer regenerate every reported result. The cell below lists what should exist and flags anything missing.

In [ ]:
expected = [
    "data/raw/primary_care_raw.csv", "data/raw/data_dictionary.csv",
    "data/processed/analysis_set.csv", "outputs/tables/cleaning_log.csv",
    "outputs/tables/analysis_set_flow.csv", "outputs/tables/table1.md", "outputs/tables/table2.md",
    "outputs/figures/figure1_predicted_probability.png", "outputs/logs/session_info.csv",
    "outputs/logs/ai_use_log.csv", "reports/_variables.yml", "reports/mini_report.qmd",
    "reports/mini_report.html",
]
manifest = pd.DataFrame({"file": expected, "present": [(ROOT / f).exists() for f in expected]})
manifest.to_csv(ROOT / "outputs" / "handover_manifest.csv", index=False)
manifest

Anything marked `False` needs attention before you submit. The capstone rubric is in the participant workbook.

## Key points

* Restart and Run All is the minimum test of a reproducible notebook.
* Write results to files from code; let the report read them.
* Table 1 describes; Table 2 answers the question; the figure shows size and uncertainty.
* Discussion: Direct answer, Literature, Framework, Alternatives and limitations, Recommendation.
* Every factual claim needs a source you have checked yourself.